# Maximum k-vertex cover with native FlagQuantum synthesis

Select exactly five nodes of the original ten-node,21-edge graph. Minimize
uncovered edges; apply a penalty of10*(selected_count-5)**2 during quantum
optimization. Keep three quantum layers,CVaR0.7,90iterations and1000 actual
samples per optimized/uniform distribution. Angles are radians.


In [ ]:
import matplotlib.pyplot as plt

import flagquantum as fq
from examples.synthesis.applications.optimization.max_k_vertex_cover.main import (
    cost_layer,
)
from examples.synthesis.applications.optimization.max_k_vertex_cover.overview.plots import (
    cover_figure,
)
from examples.synthesis.applications.optimization.max_k_vertex_cover.overview.workflow import (
    model_table,
    run_workflow,
    source_edges,
)
from examples.synthesis.graph_optimization import INITIAL_PARAMETERS, selected_nodes
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
)

print("Original graph", source_edges())
print("Nodes", 10, "edges", len(source_edges()), "K", 5)


@qfunc
def cover_program(params: list[float], v: Output[QArray[QBit]]) -> None:
    allocate(10, v)
    hadamard_transform(v)
    for layer in range(3):
        cost_layer(params[layer], v)
        for index in range(10):
            RX(params[3 + layer], v[index])


## Synthesize a native program with explicit budgets

The existing native cost helper expresses uncovered edges and the squared
cardinality penalty. Preparation,cost layers and mixers compose in one qfunc.
Named item i is bit i of v; physical wire numbers are unnecessary here.


In [ ]:
initial_program = synth.synthesize(
    cover_program,
    classical_inputs={"params": INITIAL_PARAMETERS},
    max_qubits=10,
    max_workspace=0,
    max_operations=5000,
    memory_limit_bytes=4 * 1024**2,
)
initial_readout = dict(
    initial_program.probabilities(fq.run(initial_program), registers=("v",))
)
print("Named assignments", len(initial_readout))
print("Resources", initial_program.report["resources"])


## Optimize and sample actual distributions

Every evaluation synthesizes and executes. Public native Nelder-Mead replaces
the source private optimizer; exact local probabilities replace its shot estimator
inside CVaR0.7. Keep90iterations,report actual evaluations andconvergence.
Final observations are1000 optimized CPU samples(seed2718) and1000 uniform
samples(seed31415). Preserve objective,penalty andfeasibility as separate fields.


In [ ]:
workflow = run_workflow()
print("Initial CVaR", workflow.trace[0].cost)
print("Final CVaR", workflow.optimizer.value)
print("Iterations", workflow.optimizer.iterations)
print("Actual evaluations", workflow.optimizer.evaluations)
print("Converged", workflow.optimizer.converged)
for sample in workflow.samples[:5]:
    print(sample)
print("Optimized observations", sum(row.count for row in workflow.samples))
print("Uniform observations", sum(row.count for row in workflow.uniform))
print(
    "Feasible optimized observations",
    sum(row.count for row in workflow.samples if row.feasible),
)
print(
    "Feasible uniform observations",
    sum(row.count for row in workflow.uniform if row.feasible),
)


## Inspect the best actually observed feasible cover

Require exactly five selected nodes. If no feasible assignment is observed,
report None; do not insert the exact classical answer. The figure retains the
original graph,cost trace,40-bin histogram andobserved/reference graph roles.
Spring layout(seed2718) explicitly replaces source Kamada-Kawai. Gold edges
have at least one selected endpoint; red nodes are selected.


In [ ]:
best = workflow.best_feasible
print("Best observed feasible", best)
if best is not None:
    print("Selected nodes", selected_nodes(best.word, 10))
    print("Covered edges", len(source_edges()) - best.uncovered)
    print("Uncovered edges", best.uncovered)
    print("Penalized cost", best.penalized_cost)
cover_plot = cover_figure(workflow)
plt.show()
plt.close(cover_plot)


## Separate bounded reference after sampling

Enumerating the252 feasible assignments replaces external Couenne. The reference
never supplies optimizer parameters or observed answers. Seeing an optimal cover
in a finite sample does not prove optimizer convergence or quantum advantage.


In [ ]:
exact_word = workflow.exact_reference_word
exact_uncovered, exact_penalized, exact_feasible = model_table()[exact_word]
print("Exact bounded reference nodes", selected_nodes(exact_word, 10))
print("Exact reference uncovered edges", exact_uncovered)
print("Exact reference feasible", exact_feasible)
print("Actual best feasible observation", workflow.best_feasible)
